# DanyAI на Colab

Рантайм: GPU (T4). Ячейки по порядку.

In [ ]:
!pip -q install numpy
!git clone https://github.com/DanyaVoredom/DanyAI.git
%cd DanyAI
!pip -q install -e .

In [ ]:
import os
os.makedirs("data/raw", exist_ok=True)
ids = [1342, 11, 1661, 2701, 84, 98, 345, 5200, 16328, 2600]
for i in ids:
    !curl -s -L https://www.gutenberg.org/cache/epub/{i}/pg{i}.txt -o data/raw/pg{i}.txt
!du -sh data/raw

In [ ]:
!python -m danyai.prepare --input data/raw --out data --vocab 8192 --val 0.02

In [ ]:
import json
cfg = json.load(open("configs/mini.json"))
cfg.update(block_size=256, batch_size=48, steps=20000, lr=0.0015, warmup=500)
cfg["name"] = "DanyAI-Mini-Colab"
json.dump(cfg, open("configs/colab.json", "w"), indent=2)
print(cfg)

In [ ]:
!python -m danyai.train --config configs/colab.json --data data --out runs/colab

In [ ]:
!python -m danyai.sample --ckpt runs/colab/best.npz --tok data/tokenizer.json --prompt "The house" --tokens 200 --temp 0.8